# 02 — Google Colab: Phase 4 blinded pairwise judge

This notebook loads the stronger **Qwen2.5-14B-Instruct** judge in 4-bit and evaluates paired baseline/adaptive responses from both 7B generators. X/Y order is reversed across judge repeats to measure position sensitivity.

In [ ]:
REPO_URL = "https://github.com/YOUR_USERNAME/political-bias-lab.git"
PROFILE = "smoke"
DRIVE_ROOT = "/content/drive/MyDrive/political-bias-lab"
REPO_DIR = "/content/political-bias-lab"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os, subprocess
if not os.path.exists(REPO_DIR):
    subprocess.check_call(["git", "clone", REPO_URL, REPO_DIR])
else:
    subprocess.check_call(["git", "-C", REPO_DIR, "pull", "--ff-only"])
os.chdir(REPO_DIR)
GIT_SHA = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
RUN_ID = f"{PROFILE}-{GIT_SHA[:8]}"
RUN_ROOT = f"{DRIVE_ROOT}/runs/{RUN_ID}"
print("Run ID:", RUN_ID)

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"])

In [ ]:
from src.cloud import link_colab_persistent_dirs
link_colab_persistent_dirs(REPO_DIR, RUN_ROOT)

In [ ]:
from pathlib import Path
import pandas as pd
from src.config import load_config
from src.runtime import require_colab_gpu

cfg = load_config(Path(REPO_DIR)/"config/default.yaml", Path(REPO_DIR)/f"config/{PROFILE}.yaml")
judge_cfg = cfg["models"]["judge"]
info = require_colab_gpu(float(judge_cfg.get("min_gpu_vram_gb", 14)))
print(info)
gen_path = Path(REPO_DIR)/"results/raw/phase4_generations.parquet"
assert gen_path.exists(), "No Phase 4 generations found. Run both generator sessions first."
gens = pd.read_parquet(gen_path)
print(gens.groupby(["model", "condition"])["case_id"].nunique())

In [ ]:
from src.pipeline import run_judge
from src.io_utils import write_json
from pathlib import Path
metadata = run_judge(cfg, root=REPO_DIR)
write_json(metadata, Path(REPO_DIR)/"results/manifests/judge_model.json")
metadata

In [ ]:
import pandas as pd
from pathlib import Path
p = Path(REPO_DIR)/"results/raw/phase4_judgments.parquet"
df = pd.read_parquet(p)
print("Rows:", len(df))
print("Parse success rate:", float(df["parse_success"].fillna(False).mean()))
print(df.groupby("generator_model")["parse_success"].mean())

### Important

A high parse rate alone does not validate the judge. Notebook 03 also calculates **X/Y order consistency**. If the judge fails either predeclared gate, treat the automated judge as unreliable and make blinded human evaluation the primary Phase 4 evidence.